# PaddleOCR-VL contre Tesseract — sur des scans réels du SGG

Une question reste ouverte depuis le début du projet BLDP : **PaddleOCR-VL lit-il vraiment mieux que Tesseract les documents juridiques béninois ?**

Les classements 2026 donnent 96,3 contre ~85 sur OmniDocBench — mais sur des corpus internationaux, jamais sur des scans du Secrétariat Général du Gouvernement. Ce carnet y répond avec vos documents, gratuitement, sur le GPU de Colab.

## La méthode

**24 documents choisis parmi les pires du corpus** — scores qualité de 0,22 à 0,51, et pour vingt d'entre eux **zéro article détecté** par la chaîne actuelle. Un test sur des scans propres ne prouverait rien : les deux moteurs y réussiraient.

Les deux moteurs tournent **ici, sur les mêmes images**. Comparer un résultat Colab à un résultat produit ailleurs mêlerait la différence de moteur à celle des machines.

**Aucun accès à vos serveurs n'est nécessaire** : les PDF sont publics et téléchargés directement depuis `sgg.gouv.bj`.

## Ce qui décidera

Trois mesures, dans l'ordre d'importance :

1. **Les en-têtes d'article retrouvés.** Un texte juridique a des articles. Ne pas les voir, c'est rater le document.
2. **La validité lexicale française**, mesurée au dictionnaire Hunspell. C'est ce qui sépare `Article premier` de `Articl'6 promi\ufffdr`.
3. **Le volume de texte**, en dernier — un moteur bavard qui produit du bruit n'est pas meilleur.

> **Avant de lancer** : *Exécution → Modifier le type d'exécution → GPU (T4)*.

## 1. Vérifier le GPU

PaddleOCR-VL exige CUDA. Sans GPU, ce carnet tournerait sur processeur — beaucoup plus lentement, et la comparaison de vitesse perdrait son sens.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "AUCUN GPU — activez-le : Exécution > Modifier le type d'exécution > T4"

/bin/bash: line 1: nvidia-smi: command not found
AUCUN GPU — activez-le : Exécution > Modifier le type d'exécution > T4


## 2. Installer les deux moteurs

Cinq à dix minutes. Tesseract vient d'`apt`, PaddleOCR de `pip`.

Le dictionnaire français (`hunspell-fr`) sert à la mesure de validité lexicale : sans lui, on ne saurait pas distinguer un texte lisible d'une suite de caractères plausibles.

In [ ]:
# --- Systeme : Tesseract, dictionnaire francais, outils PDF ---------------
# Sortie VISIBLE a dessein. La version precedente masquait ce bloc, et un
# echec d'installation passait inapercu jusqu'a planter dix cellules plus loin.
!apt-get -qq update > /dev/null
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra poppler-utils hunspell hunspell-fr 2>&1 | tail -2
!pip -q install pymupdf 2>&1 | tail -2
print('systeme installe')

: 

In [ ]:
# --- PaddlePaddle ----------------------------------------------------------
#
# PaddlePaddle n'est PAS sur PyPI standard : « pip install paddlepaddle-gpu »
# echoue avec un message trompeur. Il faut son index officiel, et la version
# de CUDA doit correspondre a celle du GPU alloue par Colab.
import subprocess, sys, re

info = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
trouve = re.search('CUDA Version:[ ]*([0-9]+)[.]([0-9]+)', info)
cuda = 'cu' + trouve.group(1) + trouve.group(2) if trouve else 'cu126'
print('Python', sys.version.split()[0], '| CUDA detecte :', cuda)

index, vus = [], set()
for c in [cuda, 'cu126', 'cu123', 'cu118']:
    if c not in vus:
        vus.add(c)
        index.append(c)

installe = False
for c in index:
    url = 'https://www.paddlepaddle.org.cn/packages/stable/' + c + '/'
    print()
    print('--- tentative', c, '---')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'paddlepaddle-gpu', '-i', url],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print('  installe depuis', c)
        installe = True
        break
    derniere = (r.stderr.strip().splitlines() or ['echec sans message'])[-1]
    print(' ', derniere[:160])

if not installe:
    print()
    print('--- repli : version processeur ---')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle'],
                       capture_output=True, text=True)
    installe = r.returncode == 0
    print(' ', 'installe (sans GPU : ce sera lent)' if installe else 'echec')

print()
print('paddlepaddle installe :', installe)


: 

In [ ]:
# --- PaddleOCR, puis VERIFICATION QUI ECHOUE BRUYAMMENT --------------------
!pip -q install paddleocr 2>&1 | tail -3

import subprocess
erreurs = []

try:
    import paddle
    avec_cuda = paddle.device.is_compiled_with_cuda()
    print('paddle    :', paddle.__version__, '| compile avec CUDA :', avec_cuda)
    if avec_cuda:
        print('            GPU visibles :', paddle.device.cuda.device_count())
except Exception as exc:
    erreurs.append('paddle absent : ' + type(exc).__name__ + ' ' + str(exc)[:120])

try:
    import paddleocr
    print('paddleocr :', getattr(paddleocr, '__version__', 'version inconnue'))
    print('            PaddleOCRVL disponible :', hasattr(paddleocr, 'PaddleOCRVL'))
except Exception as exc:
    erreurs.append('paddleocr absent : ' + type(exc).__name__ + ' ' + str(exc)[:120])

v = subprocess.run(['tesseract', '--version'], capture_output=True, text=True).stdout
print('tesseract :', v.splitlines()[0] if v else 'ABSENT')
dico = subprocess.run(['bash', '-c', 'ls /usr/share/hunspell/fr* 2>/dev/null'],
                      capture_output=True, text=True).stdout.strip()
print('dico fr   :', 'present' if dico else 'ABSENT')

if erreurs:
    print()
    print('=' * 70)
    for e in erreurs:
        print('  ECHEC :', e)
    print('=' * 70)
    raise RuntimeError('Installation incomplete — inutile de continuer. '
                       'Copiez la sortie ci-dessus : elle dit ce qui manque.')

print()
print('Tout est en place.')


: 

## 3. Télécharger l'échantillon depuis le SGG

Les 24 documents les plus abîmés du lot 1, avec le score et le nombre d'articles que la chaîne actuelle en a tirés — c'est la référence à battre.

Une seconde de pause entre les téléchargements : le SGG n'a pas à subir une rafale.

In [4]:
ECHANTILLON = [
    # id                        titre                                    score  pages  articles
    ("loi-2005%E2%80%9301",     "Loi 2005-01 du 12 janv. 2005",          0.224, 2, 0),
    ("ordonnance-1963-2",       "Ordonnance 1963-2 du 04 nov. 1963",     0.279, 1, 0),
    ("loi-93-015",              "Loi 93-015 du 28 sept. 1993",           0.337, 1, 0),
    ("loi-96-020",              "Loi 96-020 du 13 août 1996",            0.354, 2, 0),
    ("ordonnance-1974-36",      "Ordonnance 1974-36 du 24 avril 1974",   0.358, 5, 0),
    ("loi-94-025",              "Loi 94-025 du 16 déc. 1994",            0.365, 2, 0),
    ("ordonnance-1968-55",      "Ordonnance 1968-55 du 15 nov. 1968",    0.410, 2, 0),
    ("loi-95-001",              "Loi 95-001 du 19 août 1995",            0.415, 1, 0),
    ("loi-1995-1",              "Loi 1995-1 du 18 août 1995",            0.428, 1, 0),
    ("ordonnance-1993-04",      "Ordonnance 1993-04 du 10 mars 1993",    0.436, 2, 0),
    ("ordonnance-1978-10",      "Ordonnance 1978-10 du 23 févr. 1978",   0.436, 2, 0),
    ("ordonnance-1974-5",       "Ordonnance 1974-5 du 01 févr. 1974",    0.453, 2, 0),
    ("loi-94-006",              "Loi 94-006 du 22 juin 1994",            0.465, 2, 0),
    ("ordonnance-1976-41",      "Ordonnance 1976-41 du 15 juil. 1976",   0.470, 1, 0),
    ("ordonnance-1979-7",       "Ordonnance 1979-7 du 22 janv. 1979",    0.472, 2, 0),
    ("ordonnance-1979-30",      "Ordonnance 1979-30 du 15 mai 1979",     0.473, 2, 0),
    ("loi-1996-007",            "Loi 1996-007 du 30 mai 1996",           0.480, 2, 0),
    ("loi-2006-21",             "Loi 2006-21 du 05 déc. 2006",           0.490, 4, 2),
    ("loi-2006-23",             "Loi 2006-23 du 14 déc. 2006",           0.490, 4, 2),
    ("loi-2011-28",             "Loi 2011-28 du 18 nov. 2011",           0.490, 4, 2),
    ("loi-2011-29",             "Loi 2011-29 du 18 nov. 2011",           0.490, 4, 2),
    ("loi-2012-18",             "Loi 2012-18 du 15 mai 2012",            0.490, 4, 2),
    ("loi-1997-021",            "Loi 1997-021 du 20 juin 1997",          0.507, 2, 0),
    ("loi-97-021",              "Loi 97-021 du 20 juin 1997",            0.509, 2, 0),
]

import pathlib, time, urllib.request

PDF = pathlib.Path("pdf"); PDF.mkdir(exist_ok=True)
AGENT = "BLDP-eval/0.1 (evaluation OCR; contact dikdokmoney@gmail.com)"

recus, manquants = [], []
for ident, titre, score, pages, articles in ECHANTILLON:
    cible = PDF / f"{ident.replace('%', '_')}.pdf"
    if cible.exists() and cible.stat().st_size > 1000:
        recus.append((ident, cible, titre, score, articles)); continue
    url = f"https://sgg.gouv.bj/doc/{ident}/download"
    try:
        requete = urllib.request.Request(url, headers={"User-Agent": AGENT})
        with urllib.request.urlopen(requete, timeout=60) as reponse:
            cible.write_bytes(reponse.read())
        recus.append((ident, cible, titre, score, articles))
        print(f"  {ident:<26} {cible.stat().st_size // 1024:>5} Ko")
    except Exception as exc:
        manquants.append((ident, str(exc)[:60]))
        print(f"  {ident:<26} ÉCHEC : {exc}")
    time.sleep(1)  # politesse envers le SGG

print(f"\n{len(recus)} document(s) reçus, {len(manquants)} manquant(s)")

  loi-2005%E2%80%9301           32 Ko
  ordonnance-1963-2            658 Ko
  loi-93-015                    38 Ko
  loi-96-020                    59 Ko
  ordonnance-1974-36           161 Ko
  loi-94-025                    76 Ko
  ordonnance-1968-55            98 Ko
  loi-95-001                    46 Ko
  loi-1995-1                    34 Ko
  ordonnance-1993-04           402 Ko
  ordonnance-1978-10           493 Ko
  ordonnance-1974-5             68 Ko
  loi-94-006                    49 Ko
  ordonnance-1976-41            43 Ko
  ordonnance-1979-7             62 Ko
  ordonnance-1979-30            52 Ko
  loi-1996-007                  43 Ko
  loi-2006-21                   80 Ko
  loi-2006-23                   76 Ko
  loi-2011-28                   63 Ko
  loi-2011-29                   59 Ko
  loi-2012-18                   48 Ko
  loi-1997-021                  41 Ko
  loi-97-021                    53 Ko

24 document(s) reçus, 0 manquant(s)


: 

## 4. Rendre les pages en images

300 ppp, la même résolution que la chaîne BLDP. Les deux moteurs recevront **exactement les mêmes images** : toute différence de résultat viendra du moteur, pas du rendu.

In [5]:
import pymupdf

IMG = pathlib.Path("pages"); IMG.mkdir(exist_ok=True)
pages_par_document = {}

for ident, chemin, titre, score, articles in recus:
    try:
        document = pymupdf.open(chemin)
    except Exception as exc:
        print(f"  {ident} : PDF illisible ({exc})"); continue
    chemins = []
    with document:
        for numero in range(document.page_count):
            image = IMG / f"{ident.replace('%', '_')}_p{numero + 1}.png"
            if not image.exists():
                document[numero].get_pixmap(dpi=300).save(image)
            chemins.append(image)
    pages_par_document[ident] = chemins

total = sum(len(v) for v in pages_par_document.values())
print(f"{len(pages_par_document)} document(s), {total} page(s) rendues à 300 ppp")

24 document(s), 56 page(s) rendues à 300 ppp


: 

## 5. Les outils de mesure

**Les en-têtes d'article** sont cherchés avec un motif tolérant aux dégâts d'OCR — `Arlicle`, `ArtIcle`, `A,rUi:cfJE` doivent compter. Sinon on mesurerait la propreté du texte deux fois.

**La validité lexicale** interroge Hunspell mot à mot. C'est la mesure la plus honnête : un OCR qui produit du charabia plausible chute, un OCR qui lit vraiment monte.

In [6]:
import re, subprocess

# Tolérant à l'OCR : « Article », « Arlicle », « ArtIcle », « Adicle »…
EN_TETE_ARTICLE = re.compile(
    r"\b[Aa][rnu][^\W\d_]{0,2}[il1I][^\W\d_]{0,2}[le1I]e?\s*[:.\-–]?\s*"
    r"(?:premier|1er|[0-9]{1,3}|[IVXL]{1,6})\b",
    re.IGNORECASE,
)
MOT = re.compile(r"[A-Za-zÀ-ÿ]{4,}")

def articles_trouves(texte: str) -> int:
    return len(EN_TETE_ARTICLE.findall(texte or ""))

def validite_francaise(texte: str) -> float:
    """Part des mots de 4 lettres ou plus reconnus par le dictionnaire.

    Hunspell est interrogé en une seule fois : mot à mot, le coût de
    lancement dépasserait de loin celui de l'analyse.
    """
    mots = MOT.findall(texte or "")
    if not mots:
        return 0.0
    entree = "\n".join(mots[:4000])
    try:
        sortie = subprocess.run(
            ["hunspell", "-d", "fr_FR", "-l"],
            input=entree, capture_output=True, text=True, timeout=120,
        ).stdout
    except Exception:
        return float("nan")
    inconnus = len([l for l in sortie.splitlines() if l.strip()])
    examines = len(mots[:4000])
    return round((examines - inconnus) / examines, 4) if examines else 0.0

# Vérification du dispositif de mesure sur deux textes connus.
propre = "Article premier : la présente loi fixe les règles applicables aux marchés publics."
abime = "Articl'6 promi\ufffdr : F-rr appircalion des drsposilrons des artr\ufffdles"
print(f"texte propre  -> articles {articles_trouves(propre)}, validité {validite_francaise(propre):.2f}")
print(f"texte abîmé   -> articles {articles_trouves(abime)}, validité {validite_francaise(abime):.2f}")

texte propre  -> articles 1, validité 1.00
texte abîmé   -> articles 0, validité 0.00


: 

## 6. Tesseract

Un seul appel par page produisant `txt` **et** `tsv` — c'est la correction apportée à BLDP le 4 septembre, qui a divisé par deux le coût de l'OCR.

In [7]:
import time

resultats_tesseract, duree_tesseract = {}, 0.0
for ident, chemins in pages_par_document.items():
    morceaux = []
    depart = time.time()
    for image in chemins:
        base = image.with_suffix("")
        subprocess.run(["tesseract", str(image), str(base), "-l", "fra", "txt"],
                       capture_output=True, timeout=300)
        fichier = base.with_suffix(".txt")
        if fichier.exists():
            morceaux.append(fichier.read_text(encoding="utf-8", errors="replace"))
    duree_tesseract += time.time() - depart
    resultats_tesseract[ident] = "\n".join(morceaux)

pages_totales = sum(len(v) for v in pages_par_document.values())
print(f"Tesseract : {pages_totales} pages en {duree_tesseract:.0f} s "
      f"({duree_tesseract / max(pages_totales, 1):.1f} s/page)")

Tesseract : 56 pages en 351 s (6.3 s/page)


: 

## 7. PaddleOCR sur GPU

La cellule tente d'abord **PaddleOCR-VL**, le modèle vision-langage qui domine OmniDocBench. S'il n'est pas disponible dans la version installée, elle se rabat sur le PaddleOCR classique — et **le dit**. Un carnet qui prétendrait tester VL en testant autre chose fausserait toute la décision.

In [8]:
moteur_utilise = None
predire = None

# 1. PaddleOCR-VL, le modèle visé.
try:
    from paddleocr import PaddleOCRVL
    _vl = PaddleOCRVL()
    def predire(chemin):
        sortie = _vl.predict(str(chemin))
        textes = []
        for page in sortie:
            brut = page.get("markdown") if isinstance(page, dict) else getattr(page, "markdown", None)
            if isinstance(brut, dict):
                brut = brut.get("markdown_texts") or brut.get("text")
            textes.append(brut if isinstance(brut, str) else str(page))
        return "\n".join(textes)
    moteur_utilise = "PaddleOCR-VL"
except Exception as exc:
    print(f"PaddleOCR-VL indisponible ({type(exc).__name__}: {str(exc)[:90]})")

# 2. Repli : PaddleOCR classique.
if predire is None:
    from paddleocr import PaddleOCR
    _ocr = PaddleOCR(lang="fr")
    def predire(chemin):
        sortie = _ocr.predict(str(chemin)) if hasattr(_ocr, "predict") else _ocr.ocr(str(chemin))
        lignes = []
        for bloc in (sortie or []):
            if isinstance(bloc, dict) and "rec_texts" in bloc:
                lignes.extend(bloc["rec_texts"])
            elif isinstance(bloc, list):
                for entree in bloc:
                    if isinstance(entree, (list, tuple)) and len(entree) > 1:
                        contenu = entree[1]
                        lignes.append(contenu[0] if isinstance(contenu, (list, tuple)) else str(contenu))
        return "\n".join(lignes)
    moteur_utilise = "PaddleOCR classique (VL indisponible)"

print(f"Moteur retenu : {moteur_utilise}")

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


PaddleOCR-VL indisponible (RuntimeError: A dependency error occurred during pipeline creation. Please refer to the installation doc)


Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

RuntimeError: Engine 'paddle_static' is unavailable because dependency 'paddlepaddle' is not installed.

: 

In [ ]:
resultats_paddle, duree_paddle = {}, 0.0
for ident, chemins in pages_par_document.items():
    morceaux = []
    depart = time.time()
    for image in chemins:
        try:
            morceaux.append(predire(image))
        except Exception as exc:
            print(f"  {ident} {image.name} : {type(exc).__name__} {str(exc)[:70]}")
    duree_paddle += time.time() - depart
    resultats_paddle[ident] = "\n".join(morceaux)

print(f"{moteur_utilise} : {pages_totales} pages en {duree_paddle:.0f} s "
      f"({duree_paddle / max(pages_totales, 1):.1f} s/page)")

: 

## 8. Le verdict, document par document

In [ ]:
import pandas as pd

reference = {ident: (titre, score, articles) for ident, _, titre, score, articles in recus}
lignes = []
for ident in pages_par_document:
    titre, score, articles_bldp = reference[ident]
    t, p = resultats_tesseract.get(ident, ""), resultats_paddle.get(ident, "")
    lignes.append({
        "document": titre[:34],
        "art. BLDP": articles_bldp,
        "art. Tess": articles_trouves(t),
        "art. Paddle": articles_trouves(p),
        "fr Tess": validite_francaise(t),
        "fr Paddle": validite_francaise(p),
        "car. Tess": len(t),
        "car. Paddle": len(p),
    })

tableau = pd.DataFrame(lignes)
pd.set_option("display.width", 200, "display.max_columns", 20)
tableau

: 

In [ ]:
art_t, art_p = tableau["art. Tess"].sum(), tableau["art. Paddle"].sum()
fr_t, fr_p = tableau["fr Tess"].mean(), tableau["fr Paddle"].mean()
muets_t = int((tableau["art. Tess"] == 0).sum())
muets_p = int((tableau["art. Paddle"] == 0).sum())
n = len(tableau)

print(f"{'':<26}{'Tesseract':>12}{'Paddle':>12}")
print("-" * 50)
print(f"{'articles trouvés':<26}{art_t:>12}{art_p:>12}")
print(f"{'validité française':<26}{fr_t:>12.3f}{fr_p:>12.3f}")
print(f"{'documents muets':<26}{muets_t:>12}{muets_p:>12}   sur {n}")
print(f"{'secondes par page':<26}{duree_tesseract/max(pages_totales,1):>12.1f}"
      f"{duree_paddle/max(pages_totales,1):>12.1f}")
print()

gain_art = (art_p - art_t) / art_t if art_t else float("inf")
gain_fr = fr_p - fr_t
if gain_art > 0.30 or gain_fr > 0.10:
    verdict = ("NET : Paddle justifie de louer une instance GPU.\n"
               "   Coût estimé pour les décrets historiques : environ 10 $.")
elif gain_art < 0.05 and gain_fr < 0.03:
    verdict = ("AUCUN : gardez Tesseract.\n"
               "   Vous économisez le budget GPU ET la complexité d'une seconde chaîne.")
else:
    verdict = ("MODESTE : à trancher au cas par cas.\n"
               "   Paddle pourrait ne servir qu'aux documents que Tesseract laisse muets.")
print("VERDICT —", verdict)
print(f"\n(articles {gain_art:+.0%}, validité française {gain_fr:+.3f})")
print(f"Moteur réellement testé : {moteur_utilise}")

: 

## 9. Lire les textes soi-même

Les mesures orientent, elles ne remplacent pas la lecture. Sur un corpus juridique, c'est le regard qui tranche.

In [ ]:
# Les trois documents où l'écart est le plus grand.
tableau["ecart"] = tableau["art. Paddle"] - tableau["art. Tess"]
for rang in tableau.sort_values("ecart", ascending=False).head(3).index:
    ident = list(pages_par_document)[rang]
    titre = reference[ident][0]
    print("=" * 78)
    print(titre)
    print("=" * 78)
    print("\n--- TESSERACT ---")
    print((resultats_tesseract.get(ident, "") or "(vide)")[:620])
    print(f"\n--- {moteur_utilise.upper()} ---")
    print((resultats_paddle.get(ident, "") or "(vide)")[:620])
    print()

: 

## 10. Emporter les résultats

Une session Colab est éphémère. Ce fichier conserve les textes des deux moteurs et les mesures — de quoi rejouer la comparaison sans refaire tourner le GPU.

In [ ]:
import json, datetime

rapport = {
    "produit_le": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "moteur_compare": moteur_utilise,
    "documents": len(pages_par_document),
    "pages": pages_totales,
    "secondes_par_page": {
        "tesseract": round(duree_tesseract / max(pages_totales, 1), 2),
        "paddle": round(duree_paddle / max(pages_totales, 1), 2),
    },
    "totaux": {
        "articles_tesseract": int(art_t), "articles_paddle": int(art_p),
        "validite_tesseract": round(float(fr_t), 4), "validite_paddle": round(float(fr_p), 4),
        "muets_tesseract": muets_t, "muets_paddle": muets_p,
    },
    "par_document": [
        {
            "id": ident,
            "titre": reference[ident][0],
            "score_bldp": reference[ident][1],
            "texte_tesseract": resultats_tesseract.get(ident, ""),
            "texte_paddle": resultats_paddle.get(ident, ""),
        }
        for ident in pages_par_document
    ],
}

chemin = "comparaison_ocr_sgg.json"
with open(chemin, "w", encoding="utf-8") as fichier:
    json.dump(rapport, fichier, ensure_ascii=False, indent=1)
print(f"écrit : {chemin} ({pathlib.Path(chemin).stat().st_size // 1024} Ko)")

try:
    from google.colab import files
    files.download(chemin)
except Exception:
    print("(hors Colab : récupérez le fichier depuis le panneau des fichiers)")

: 